In [ ]:
import numpy as np
import xarray as xr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
filename = "assets/Voxel_Schneckenstein_II_10x10.nc"
fieldName = "Schneckenstein_II_Prediction_0"

In [ ]:
# Daten laden
# NetCDF-Datei öffnen und das 3D-Voxelgitter extrahieren
ds = xr.open_dataset(filename)
data = ds[fieldName]

# Werte als NumPy-Array holen
vol = data.values
# NaN-Werte werden durch -1 ersetzt und in Integer konvertiert
vol = np.nan_to_num(vol, nan=-1).astype(int)

# Koordinaten entlang jeder Achse auslesen
x_coords = data['x'].values
y_coords = data['y'].values
z_coords = data['z'].values

# Dimensionen des Volumens
nz, ny, nx = vol.shape

In [ ]:
# Gesteinsklassen 
## alle eindeutigen Klassen-IDs im Volumen ermitteln 
classes = np.unique(vol)
classes = classes[classes >= 0]
n_classes = len(classes)

# Farbpalette für die KLassen
palette = px.colors.qualitative.Plotly
def class_color(i):
    return palette[i % len(palette)]

# Koordinatengitter erstellen
# indexing='ij' -> erste Achse = x, zweite = y, dritte = z
X3, Y3, Z3 = np.meshgrid(x_coords, y_coords, z_coords, indexing='ij')

# vol transponieren
vol_xyz = np.transpose(vol, (2, 1, 0))  # (nx, ny, nz)

# Startebene für den Schnitt
k0 = nz // 2

In [ ]:
# Subplots: 3D links, Heatmap rechts
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=("3D Volumen", f"XY-Schnitt  z = {z_coords[k0]:.1f} m"),
    column_widths=[0.65, 0.35],
    horizontal_spacing=0.05,
)

# 3D: Scatter3d pro Klasse
# Für jede Klasse werden die Voxel-Koordinaten als Scatter3d-Punkte gezeichnet.
# Jede Klasse bekommt ihre eigene Farbe und einen Legendeneintrag.
for i, cls in enumerate(classes):
    mask = vol_xyz == cls
    fig.add_trace(
        go.Scatter3d(
            x=X3[mask], y=Y3[mask], z=Z3[mask],
            mode='markers',
            marker=dict(size=4, color=class_color(i), opacity=0.4),
            name=f"Klasse {cls}",
            legendgroup=f"cls{cls}", # Gruppe: verbindet 3D- und 2D-Trace
            showlegend=True,
        ),
        row=1, col=1,
    )

# 3D: Schnittebene (graue Surface)
# Eine horizontale Fläche zeigt an, welche z-Ebene gerade im 2D-Plot dargestellt wird.
# Der Trace-Index dieser Surface wird festgehalten, um sie per Slider zu verschieben.
surface_trace_idx = n_classes
Z_plane0 = np.full((ny, nx), z_coords[k0])  # Alle Werte = aktuelle Tiefe

fig.add_trace(
    go.Surface(
        x=x_coords, y=y_coords, z=Z_plane0,
        opacity=0.25, showscale=False, colorscale="Greys",
        name="Schnittebene", showlegend=False,
    ),
    row=1, col=1,
)

# 2D: eine Heatmap pro Klasse
# Für jede Klasse gibt es eine separate Heatmap, damit Sichtbarkeit per Legende
# (legendgroup) synchron mit dem 3D-Scatter gesteuert werden kann.
# Zellen anderer Klassen werden als None (= transparent) übergeben.
heatmap_trace_start = n_classes + 1   # Trace-Indizes: n_classes+1 … 2*n_classes

def make_slice(k, cls):
    """Gibt (ny, nx)-Array zurück: cls-Wert wo Klasse stimmt, sonst None."""
    slc = vol[k, :, :].astype(object)
    slc[slc != cls] = None
    return slc.tolist()

for i, cls in enumerate(classes):
    fig.add_trace(
        go.Heatmap(
            x=x_coords,
            y=y_coords,
            z=make_slice(k0, cls),
            colorscale=[[0, class_color(i)], [1, class_color(i)]],  # Einheitsfarbe
            zmin=int(cls), zmax=int(cls),
            showscale=False,
            name=f"Klasse {cls}",
            legendgroup=f"cls{cls}",   # ← gleiche Gruppe wie Scatter3d!
            showlegend=False,          # Legende nur einmal (über Scatter3d)
        ),
        row=1, col=2,
    )

fig = go.Figure(fig)   # nötig, damit restyle-Indizes sicher stimmen

# Slider: aktualisiert Surface + alle Heatmap-Traces + Subtitle
steps = []
for k in range(nz):
    z_val   = z_coords[k]
    Z_plane = np.full((ny, nx), z_val)

    # Neue z-Daten für Surface
    new_z_surface = [Z_plane]

    # Neue z-Daten für alle Heatmap-Traces
    new_z_heatmaps = [make_slice(k, cls) for cls in classes]

    # Trace-Indizes: Surface + alle Heatmaps
    trace_indices = [surface_trace_idx] + list(
        range(heatmap_trace_start, heatmap_trace_start + n_classes)
    )

    steps.append(dict(
        method="update",
        args=[
            {"z": [Z_plane] + new_z_heatmaps},   # Surface zuerst, dann Heatmaps
            {"annotations[1].text": f"XY-Schnitt  z = {z_val:.1f} m"},
            trace_indices,
        ],
        label=f"{z_val:.1f}",
    ))

sliders = [dict(
    active=k0,
    currentvalue={"prefix": "z = ", "suffix": " m", "font": {"size": 13}},
    pad={"t": 30, "b": 10},
    len=0.65,
    x=0.0,
    y=0.0,
    steps=steps,
)]

# Feste Achsen
x_range = [float(x_coords.min()), float(x_coords.max())]
y_range = [float(y_coords.min()), float(y_coords.max())]
z_range = [float(z_coords.min()), float(z_coords.max())]

fig.update_layout(
    width=1300,
    height=700,
    margin=dict(l=0, r=10, t=50, b=90),
    sliders=sliders,
    legend=dict(
        title="Gesteinsklassen<br><sup>(klick = toggle)</sup>",
        x=1.01, y=0.95,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgrey",
        borderwidth=1,
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
    scene=dict(
        xaxis=dict(title="X (m)", range=x_range),
        yaxis=dict(title="Y (m)", range=y_range),
        zaxis=dict(title="Z (m)", range=z_range),
        aspectmode="cube",
        domain=dict(x=[0.0, 0.62], y=[0.0, 1.0]),
    ),
)

# Achsenbeschriftungen für den 2D-Plot
fig.update_xaxes(title_text="X (m)", row=1, col=2)
fig.update_yaxes(title_text="Y (m)", row=1, col=2)

# Fertige interaktive Figur anzeigen
fig.show()